In [ ]:
# ============================================================
# Encoder selection and merge-group discovery
#
# Purpose:
#   1. Create one global 80/20 stratified train/val split.
#   2. Evaluate original SigLIP2 and DINO-adapted SigLIP2 checkpoints
#      using an MLP probe classifier.
#   3. Select the encoder with the best validation macro-F1.
#   4. Use the winning probe predictions to discover merge groups.
#
# Outputs for the hierarchical classifier:
#   train_df
#   val_df
#   leaf_classes
#   best_dino_ckpt_path
#   merge_groups
#   leaf_to_coarse
# ============================================================


from pathlib import Path
import random
import json
import gc
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from transformers import AutoProcessor, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from IPython.display import display


# ============================================================
# 0. Config
# ============================================================

DATA_ROOT = Path("/kaggle/input/datasets/kelkalot/the-hyper-kvasir-dataset")
LABELED_ROOT = DATA_ROOT / "labeled-images"
LABEL_CSV = LABELED_ROOT / "image-labels.csv"

OUTPUT_DIR = Path("/kaggle/working/two_stage_leaf_mlp_coarse_swin_aug")
ENCODER_SELECTION_DIR = OUTPUT_DIR / "encoder_selection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ENCODER_SELECTION_DIR.mkdir(parents=True, exist_ok=True)

SIGLIP_MODEL_NAME = "google/siglip2-base-patch16-224"

DINO_CHECKPOINTS = [
    None,  # epoch 0: original pretrained SigLIP2, no DINO checkpoint
    Path("/kaggle/input/notebooks/lilyii70/dinov2-style-siglip/siglip2_dino_style_unlabeled_full/siglip2_dino_style_epoch1.pt"),
    Path("/kaggle/input/notebooks/lilyii70/dinov2-style-siglip/siglip2_dino_style_unlabeled_full/siglip2_dino_style_epoch2.pt"),
    Path("/kaggle/input/notebooks/lilyii70/dinov2-style-siglip/siglip2_dino_style_unlabeled_full/siglip2_dino_style_epoch3.pt"),
    Path("/kaggle/input/notebooks/lilyii70/dinov2-style-siglip/siglip2_dino_style_unlabeled_full/siglip2_dino_style_epoch4.pt"),
    Path("/kaggle/input/notebooks/lilyii70/dinov2-style-siglip/siglip2_dino_style_unlabeled_full/siglip2_dino_style_epoch5.pt"),
]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RANDOM_STATE = 42
TRAIN_RATIO = 0.80
MIN_CLASS_COUNT = 100
MERGE_THRESHOLD = 15

NUM_WORKERS = 2
EMBED_BATCH_SIZE = 128

MLP_HIDDEN_DIM = 256
MLP_DROPOUT = 0.25
MLP_BATCH_SIZE = 128
MLP_EPOCHS = 100
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_PATIENCE = 15
USE_CLASS_WEIGHT = True

print("Device:", DEVICE)
print("Encoder selection output:", ENCODER_SELECTION_DIR)


# ============================================================
# 1. Seed
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)


# ============================================================
# 2. Basic helpers
# ============================================================

def clean_text(x):
    x = str(x).strip().lower()
    x = x.replace("\\", "/")
    x = x.replace("_", "-")
    x = x.replace(" ", "-")
    return "-".join([p for p in x.split("-") if p])


def make_class_weights(y_encoded, num_classes):
    counts = np.bincount(y_encoded, minlength=num_classes).astype(np.float32)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


# ============================================================
# 3. Load labeled data and create ONE global split
# ============================================================

def build_labeled_image_index():
    image_index = {}
    for p in LABELED_ROOT.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            image_index[p.stem] = str(p)
    print("Indexed labeled images:", len(image_index))
    if len(image_index) == 0:
        raise ValueError(f"No images found under {LABELED_ROOT}")
    return image_index


def load_count_gt_100_leaf_df():
    df = pd.read_csv(LABEL_CSV)

    required_cols = ["Video file", "Organ", "Classification", "Finding"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["video_file"] = df["Video file"].astype(str).str.strip()
    df["Organ"] = df["Organ"].apply(clean_text)
    df["Classification"] = df["Classification"].apply(clean_text)
    df["Finding"] = df["Finding"].apply(clean_text)
    df["full_label_reference"] = df["Organ"] + "/" + df["Classification"] + "/" + df["Finding"]
    df["cls_label"] = df["Finding"]

    image_index = build_labeled_image_index()
    df["image_path"] = df["video_file"].apply(lambda x: image_index.get(Path(x).stem))
    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

    counts = df["cls_label"].value_counts()
    keep_classes = sorted(counts[counts > MIN_CLASS_COUNT].index.tolist())
    df = df[df["cls_label"].isin(keep_classes)].reset_index(drop=True)

    count_df = df["cls_label"].value_counts().rename_axis("cls_label").reset_index(name="count").sort_values("count", ascending=False)

    print("\n========== Count > 100 leaf data ==========")
    print("Total images:", len(df))
    print("Num classes:", len(keep_classes))
    display(count_df)

    df.to_csv(OUTPUT_DIR / "all_count_gt_100_leaf_images.csv", index=False)
    count_df.to_csv(OUTPUT_DIR / "count_gt_100_leaf_class_counts.csv", index=False)

    return df, keep_classes


def make_global_split():
    df, leaf_classes = load_count_gt_100_leaf_df()

    train_df, val_df = train_test_split(
        df,
        train_size=TRAIN_RATIO,
        random_state=RANDOM_STATE,
        stratify=df["cls_label"],
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    split_counts = pd.concat([
        train_df["cls_label"].value_counts().rename("train_count"),
        val_df["cls_label"].value_counts().rename("val_count"),
    ], axis=1).fillna(0).astype(int)

    split_counts["total_count"] = split_counts["train_count"] + split_counts["val_count"]
    split_counts = split_counts.loc[leaf_classes]

    train_df.to_csv(OUTPUT_DIR / "global_train_df.csv", index=False)
    val_df.to_csv(OUTPUT_DIR / "global_val_df.csv", index=False)
    split_counts.to_csv(OUTPUT_DIR / "global_split_counts.csv")

    print("\n========== Global 80/20 split ==========")
    print("Train:", len(train_df))
    print("Val:", len(val_df))
    display(split_counts)

    return train_df, val_df, leaf_classes, split_counts


# ============================================================
# 4. SigLIP embedding dataset
# ============================================================

class SigLIPImageDataset(Dataset):
    def __init__(self, df):
        self.paths = df["image_path"].astype(str).tolist()
        self.labels = df["cls_label"].astype(str).tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        return image, self.labels[idx], self.paths[idx]


def siglip_collate_fn(batch):
    images = [x[0] for x in batch]
    labels = [x[1] for x in batch]
    paths = [x[2] for x in batch]
    return images, labels, paths


# ============================================================
# 5. Load one adapted SigLIP checkpoint
# ============================================================

def load_adapted_siglip_encoder(dino_ckpt_path=None):
    processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_NAME, trust_remote_code=True)

    model = AutoModel.from_pretrained(
        SIGLIP_MODEL_NAME,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    ).to(DEVICE)

    if dino_ckpt_path is None:
        dino_info = {
            "epoch": 0,
            "ssl_train_loss": None,
            "checkpoint_name": "original_siglip2",
            "checkpoint_path": None,
        }

        print("Checkpoint: original pretrained SigLIP2")
        print("DINO epoch: 0")
        print("SSL train loss: None")
        print("No DINO checkpoint loaded.")

    else:
        ckpt = torch.load(dino_ckpt_path, map_location="cpu", weights_only=False)

        if "student_state_dict" not in ckpt:
            raise KeyError(f"{dino_ckpt_path} does not contain student_state_dict")

        missing, unexpected = model.load_state_dict(
            ckpt["student_state_dict"],
            strict=False,
        )

        dino_info = {
            "epoch": int(ckpt.get("epoch", -1)),
            "ssl_train_loss": ckpt.get("ssl_train_loss"),
            "checkpoint_name": dino_ckpt_path.name,
            "checkpoint_path": str(dino_ckpt_path),
        }

        print("Checkpoint:", dino_ckpt_path.name)
        print("DINO epoch:", dino_info["epoch"])
        print("SSL train loss:", dino_info["ssl_train_loss"])
        print("Missing keys:", len(missing))
        print("Unexpected keys:", len(unexpected))

        del ckpt

    for p in model.parameters():
        p.requires_grad = False

    model.eval()
    gc.collect()

    return processor, model, dino_info


# ============================================================
# 6. Temporary embedding extraction for probe evaluation
# ============================================================

@torch.no_grad()
def extract_siglip_embeddings_no_save(df, processor, model, split_name):
    ds = SigLIPImageDataset(df)

    loader = DataLoader(
        ds,
        batch_size=EMBED_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        collate_fn=siglip_collate_fn,
        pin_memory=torch.cuda.is_available(),
    )

    all_embs = []

    for images, _, _ in tqdm(loader, desc=f"Extracting {split_name} embeddings"):
        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in inputs.items()}

        outputs = model.get_image_features(**inputs)
        emb = outputs.pooler_output

        if emb is None:
            raise RuntimeError("SigLIP2 did not return pooler_output")

        emb = F.normalize(emb.float(), dim=-1)
        all_embs.append(emb.cpu())

    return torch.cat(all_embs, dim=0).numpy().astype(np.float32)


# ============================================================
# 7. Probe classifier
# ============================================================

class MLPHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def predict_mlp(model, X_s, label_encoder, batch_size=1024):
    model.eval()
    all_probs = []

    for start in range(0, len(X_s), batch_size):
        end = min(start + batch_size, len(X_s))
        xb = torch.tensor(X_s[start:end], dtype=torch.float32, device=DEVICE)
        probs = F.softmax(model(xb), dim=1).cpu().numpy()
        all_probs.append(probs)

    probs = np.concatenate(all_probs, axis=0)
    pred_idx = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    pred = label_encoder.inverse_transform(pred_idx)

    return pred, conf, probs


def train_probe_classifier(X_train_s, y_train, X_val_s, y_val):
    label_encoder = LabelEncoder()
    label_encoder.fit(y_train)

    y_train_enc = label_encoder.transform(y_train)

    train_ds = TensorDataset(
        torch.tensor(X_train_s, dtype=torch.float32),
        torch.tensor(y_train_enc, dtype=torch.long),
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=MLP_BATCH_SIZE,
        shuffle=True,
        drop_last=False,
    )

    model = MLPHead(
        input_dim=X_train_s.shape[1],
        hidden_dim=MLP_HIDDEN_DIM,
        num_classes=len(label_encoder.classes_),
        dropout=MLP_DROPOUT,
    ).to(DEVICE)

    if USE_CLASS_WEIGHT:
        class_weights = make_class_weights(y_train_enc, len(label_encoder.classes_)).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)

    best_state = None
    best_epoch = -1
    best_macro_f1 = -1.0
    best_pred = None
    best_conf = None
    patience = 0

    for epoch in range(1, MLP_EPOCHS + 1):
        model.train()
        losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

        pred, conf, _ = predict_mlp(model, X_val_s, label_encoder)
        metrics = compute_metrics(y_val, pred)

        print(
            f"Probe epoch {epoch:03d} | "
            f"loss={np.mean(losses):.4f} | "
            f"acc={metrics['accuracy']:.4f} | "
            f"bal_acc={metrics['balanced_accuracy']:.4f} | "
            f"macro_f1={metrics['f1_macro']:.4f}"
        )

        if metrics["f1_macro"] > best_macro_f1:
            best_macro_f1 = float(metrics["f1_macro"])
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_pred = np.array(pred).astype(str)
            best_conf = np.array(conf).astype(float)
            patience = 0
        else:
            patience += 1

        if patience >= MLP_PATIENCE:
            print(f"Probe early stopping | best_epoch={best_epoch} | best_macro_f1={best_macro_f1:.4f}")
            break

    model.load_state_dict(best_state)
    model.eval()

    return {
        "model": model,
        "label_encoder": label_encoder,
        "best_epoch": best_epoch,
        "best_macro_f1": best_macro_f1,
        "pred": best_pred,
        "conf": best_conf,
    }



# ============================================================
# 8. Merge discovery
# ============================================================

def find_connected_components(nodes, edges):
    parent = {x: x for x in nodes}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for a, b in edges:
        union(a, b)

    groups = {}

    for x in nodes:
        root = find(x)
        groups.setdefault(root, []).append(x)

    components = [sorted(group) for group in groups.values() if len(group) >= 2]
    return sorted(components, key=lambda x: (len(x), x), reverse=True)


def make_merge_label(group):
    return "__MERGE__" + "__".join(sorted([str(x) for x in group]))


def discover_merge_groups(y_true, y_pred, leaf_classes):
    labels = list(leaf_classes)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)

    pair_rows = []
    edges = []

    for i, a in enumerate(labels):
        for j, b in enumerate(labels):
            if i >= j:
                continue

            a_to_b = int(cm[i, j])
            b_to_a = int(cm[j, i])
            mutual = a_to_b + b_to_a

            pair_rows.append({
                "class_a": a,
                "class_b": b,
                "a_to_b": a_to_b,
                "b_to_a": b_to_a,
                "mutual_confusion": mutual,
                "merge": mutual >= MERGE_THRESHOLD,
            })

            if mutual >= MERGE_THRESHOLD:
                edges.append((a, b))

    pair_df = pd.DataFrame(pair_rows).sort_values("mutual_confusion", ascending=False).reset_index(drop=True)
    merge_groups = find_connected_components(labels, edges)

    leaf_to_coarse = {leaf: leaf for leaf in labels}

    group_rows = []

    for group_id, group in enumerate(merge_groups, start=1):
        merge_label = make_merge_label(group)

        for leaf in group:
            leaf_to_coarse[leaf] = merge_label

        group_rows.append({
            "merge_group_id": group_id,
            "merge_label": merge_label,
            "num_classes": len(group),
            "classes": json.dumps(group),
        })

    merge_group_df = pd.DataFrame(group_rows)

    cm_df.to_csv(ENCODER_SELECTION_DIR / "winning_probe_confusion_matrix.csv")
    pair_df.to_csv(ENCODER_SELECTION_DIR / "winning_probe_confusion_pairs.csv", index=False)
    merge_group_df.to_csv(ENCODER_SELECTION_DIR / "merge_groups.csv", index=False)

    mapping_df = pd.DataFrame([
        {"leaf_label": leaf, "coarse_label": coarse, "is_merged": leaf != coarse}
        for leaf, coarse in leaf_to_coarse.items()
    ])

    mapping_df.to_csv(ENCODER_SELECTION_DIR / "leaf_to_coarse_mapping.csv", index=False)

    print("\n========== Winning probe merge discovery ==========")
    print("MERGE_THRESHOLD:", MERGE_THRESHOLD)
    print("\nTop confusion pairs:")
    display(pair_df.head(30))
    print("\nMerge groups:")
    display(merge_group_df)
    print("\nLeaf -> coarse mapping:")
    display(mapping_df)

    return {
        "cm_df": cm_df,
        "pair_df": pair_df,
        "merge_groups": merge_groups,
        "merge_group_df": merge_group_df,
        "leaf_to_coarse": leaf_to_coarse,
        "mapping_df": mapping_df,
    }



# ============================================================
# 9. Run encoder selection and merge-group discovery
# ============================================================

def run_encoder_selection():
    train_df, val_df, leaf_classes, split_counts = make_global_split()

    y_train_leaf = train_df["cls_label"].astype(str).values
    y_val_leaf = val_df["cls_label"].astype(str).values

    comparison_rows = []

    best_macro_f1 = -1.0
    best_dino_ckpt_path = None
    best_dino_info = None
    best_probe_pred = None
    best_probe_conf = None
    best_probe_epoch = None

    for dino_ckpt_path in DINO_CHECKPOINTS:
        if dino_ckpt_path is None:
            checkpoint_name = "original_siglip2_epoch0"
        else:
            checkpoint_name = dino_ckpt_path.name
    
        print("\n" + "=" * 100)
        print("Evaluating:", checkpoint_name)
        print("=" * 100)
    
        if dino_ckpt_path is not None and not dino_ckpt_path.exists():
            raise FileNotFoundError(f"Missing DINO checkpoint: {dino_ckpt_path}")

        set_seed(RANDOM_STATE)

        processor, encoder, dino_info = load_adapted_siglip_encoder(dino_ckpt_path)

        X_train = extract_siglip_embeddings_no_save(train_df, processor, encoder, "probe train")
        X_val = extract_siglip_embeddings_no_save(val_df, processor, encoder, "probe val")

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train).astype(np.float32)
        X_val_s = scaler.transform(X_val).astype(np.float32)

        probe = train_probe_classifier(
            X_train_s=X_train_s,
            y_train=y_train_leaf,
            X_val_s=X_val_s,
            y_val=y_val_leaf,
        )

        candidate_macro_f1 = float(probe["best_macro_f1"])

        comparison_rows.append({
            "checkpoint": None if dino_ckpt_path is None else str(dino_ckpt_path),
            "checkpoint_name": checkpoint_name,
            "dino_epoch": dino_info["epoch"],
            "ssl_train_loss": dino_info["ssl_train_loss"],
            "probe_best_epoch": probe["best_epoch"],
            "probe_val_macro_f1": candidate_macro_f1,
        })

        print(
            f"Finished {checkpoint_name} | "
            f"DINO epoch={dino_info['epoch']} | "
            f"SSL loss={dino_info['ssl_train_loss']} | "
            f"Probe best epoch={probe['best_epoch']} | "
            f"Probe val macro-F1={candidate_macro_f1:.4f}"
        )

        if candidate_macro_f1 > best_macro_f1:
            best_macro_f1 = candidate_macro_f1
            best_dino_ckpt_path = None if dino_ckpt_path is None else Path(dino_ckpt_path)
            best_dino_info = dict(dino_info)
            best_probe_pred = probe["pred"].copy()
            best_probe_conf = probe["conf"].copy()
            best_probe_epoch = int(probe["best_epoch"])

        probe["model"].cpu()
        encoder.cpu()

        del probe
        del encoder
        del processor
        del scaler
        del X_train
        del X_val
        del X_train_s
        del X_val_s

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    comparison_df = pd.DataFrame(comparison_rows).sort_values("probe_val_macro_f1", ascending=False).reset_index(drop=True)
    comparison_df.to_csv(ENCODER_SELECTION_DIR / "dino_checkpoint_comparison.csv", index=False)

    print("\n========== DINO checkpoint comparison ==========")
    display(comparison_df)

    print("\n========== Selected checkpoint ==========")
    print("Checkpoint:", best_dino_ckpt_path)
    print("DINO epoch:", best_dino_info["epoch"])
    print("SSL train loss:", best_dino_info["ssl_train_loss"])
    print("Winning probe epoch:", best_probe_epoch)
    print("Winning probe val macro-F1:", best_macro_f1)

    winning_pred_df = pd.DataFrame({
        "image_path": val_df["image_path"].astype(str).values,
        "true_label": y_val_leaf,
        "pred_label": best_probe_pred,
        "confidence": best_probe_conf,
        "correct": y_val_leaf == best_probe_pred,
    })

    winning_pred_df.to_csv(ENCODER_SELECTION_DIR / "winning_probe_predictions.csv", index=False)

    merge_info = discover_merge_groups(
        y_true=y_val_leaf,
        y_pred=best_probe_pred,
        leaf_classes=leaf_classes,
    )

    return {
        "train_df": train_df,
        "val_df": val_df,
        "leaf_classes": leaf_classes,
        "split_counts": split_counts,
        "best_dino_ckpt_path": best_dino_ckpt_path,
        "best_dino_info": best_dino_info,
        "comparison_df": comparison_df,
        "winning_probe_macro_f1": best_macro_f1,
        "winning_probe_epoch": best_probe_epoch,
        "merge_groups": merge_info["merge_groups"],
        "leaf_to_coarse": merge_info["leaf_to_coarse"],
        "merge_info": merge_info,
    }


selection_results = run_encoder_selection()

train_df = selection_results["train_df"]
val_df = selection_results["val_df"]
leaf_classes = selection_results["leaf_classes"]

best_dino_ckpt_path = selection_results["best_dino_ckpt_path"]
merge_groups = selection_results["merge_groups"]
leaf_to_coarse = selection_results["leaf_to_coarse"]

print("\n" + "=" * 100)
print("ENCODER SELECTION COMPLETE")
print("=" * 100)
if best_dino_ckpt_path is None:
    print("Best SigLIP encoder: original pretrained SigLIP2 (epoch 0)")
else:
    print("Best SigLIP encoder:", best_dino_ckpt_path)
print("Merge groups:", merge_groups)
print("The hierarchical classifier will use the selected encoder, merge mapping, and the same train/val split.")

In [ ]:
# ============================================================
# Hierarchical classifier training and evaluation
#
# Purpose:
#   1. Load the selected SigLIP2 encoder and merge-group mapping
#      produced by encoder selection.
#   2. Re-extract embeddings using the selected encoder.
#   3. Fit a new scaler and train the coarse classifier on
#      coarse labels defined by the merge-group mapping.
#   4. Train one specialist classifier for each merge group.
#   5. Route coarse merge-group predictions to the corresponding
#      specialist classifier.
#   6. Evaluate the complete hierarchical classifier at the
#      original leaf-class level.
#   7. Export the coarse head, hierarchy, specialist heads,
#      scaler, and encoder metadata as hierarchical_classifier.pt.
#
# Inputs:
#   train_df
#   val_df
#   leaf_classes
#   best_dino_ckpt_path
#   merge_groups
#   leaf_to_coarse
#
# ============================================================

import shutil
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import classification_report


# ============================================================
# 0. Model B config
# ============================================================

COARSE_DIR = OUTPUT_DIR / "coarse_classifier"
SPECIALIST_DIR = OUTPUT_DIR / "specialists"
HIERARCHICAL_DIR = OUTPUT_DIR / "hierarchical_classifier"

for d in [COARSE_DIR, SPECIALIST_DIR, HIERARCHICAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SPECIALIST_BACKBONES = [
    {"tag": "swin_tiny", "hf_model_name": "microsoft/swin-tiny-patch4-window7-224"},
]

SPECIALIST_BATCH_SIZE = 16
SPECIALIST_EPOCHS = 20
SPECIALIST_LR = 2e-5
SPECIALIST_WEIGHT_DECAY = 1e-4
SPECIALIST_PATIENCE = 6

USE_SPECIALIST_AUGMENTATION = True
MAX_COPY_WRONG_PER_PAIR = 80

specialist_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.80, 1.00)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
])


# ============================================================
# 1. Evaluation helpers
# ============================================================

def save_eval_outputs(out_dir, y_true, y_pred, y_conf, image_paths, labels_for_cm=None, prefix=""):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    metrics = compute_metrics(y_true, y_pred)
    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(out_dir / f"{prefix}metrics.csv", index=False)

    report_df = pd.DataFrame(classification_report(y_true, y_pred, output_dict=True, zero_division=0)).T.reset_index().rename(columns={"index": "label"})
    report_df.to_csv(out_dir / f"{prefix}classification_report.csv", index=False)

    pred_df = pd.DataFrame({
        "image_path": image_paths,
        "true_label": y_true,
        "pred_label": y_pred,
        "confidence": y_conf,
        "correct": np.array(y_true) == np.array(y_pred),
    })

    pred_df.to_csv(out_dir / f"{prefix}predictions.csv", index=False)

    if labels_for_cm is None:
        labels_for_cm = sorted(np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)])))

    cm = confusion_matrix(y_true, y_pred, labels=labels_for_cm)
    cm_df = pd.DataFrame(cm, index=labels_for_cm, columns=labels_for_cm)
    cm_df.to_csv(out_dir / f"{prefix}confusion_matrix.csv")

    confusion_rows = []

    for i, true_label in enumerate(labels_for_cm):
        for j, pred_label in enumerate(labels_for_cm):
            if i == j:
                continue

            count = int(cm[i, j])

            if count > 0:
                confusion_rows.append({
                    "true_label": true_label,
                    "pred_label": pred_label,
                    "count": count,
                })

    confusion_df = pd.DataFrame(confusion_rows)

    if len(confusion_df) > 0:
        confusion_df = confusion_df.sort_values("count", ascending=False).reset_index(drop=True)
    else:
        confusion_df = pd.DataFrame(columns=["true_label", "pred_label", "count"])

    confusion_df.to_csv(out_dir / f"{prefix}most_common_confusions.csv", index=False)

    wrong_df = pred_df[~pred_df["correct"]].copy().sort_values("confidence", ascending=False)
    wrong_df.to_csv(out_dir / f"{prefix}wrong_predictions_sorted_by_confidence.csv", index=False)

    return {
        "metrics": metrics,
        "metrics_df": metrics_df,
        "report_df": report_df,
        "pred_df": pred_df,
        "cm_df": cm_df,
        "confusion_df": confusion_df,
        "wrong_df": wrong_df,
    }


def copy_wrong_images(wrong_df, out_dir, max_per_pair=80):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if len(wrong_df) == 0:
        return

    for (true_label, pred_label), sub in tqdm(wrong_df.groupby(["true_label", "pred_label"]), desc=f"Copy wrong images -> {out_dir.name}"):
        sub = sub.sort_values("confidence", ascending=False).head(max_per_pair)
        pair_dir = out_dir / f"TRUE_{true_label}__PRED_{pred_label}"
        pair_dir.mkdir(parents=True, exist_ok=True)

        for _, row in sub.iterrows():
            src = Path(row["image_path"])
            conf = float(row["confidence"])
            dst = pair_dir / f"conf_{conf:.3f}__{src.name}"

            if not dst.exists():
                shutil.copy2(src, dst)


# ============================================================
# 2. Model B label helper
# ============================================================

def add_coarse_labels(df, leaf_to_coarse):
    out = df.copy()
    out["coarse_label"] = out["cls_label"].map(leaf_to_coarse)

    if out["coarse_label"].isna().any():
        missing = out.loc[out["coarse_label"].isna(), "cls_label"].unique().tolist()
        raise ValueError(f"Missing coarse mapping for: {missing}")

    return out


# ============================================================
# 3. Fresh embedding extraction for Model B
# ============================================================

@torch.no_grad()
def extract_coarse_embeddings(df, processor, model, split_name):
    ds = SigLIPImageDataset(df)

    loader = DataLoader(
        ds,
        batch_size=EMBED_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        collate_fn=siglip_collate_fn,
        pin_memory=torch.cuda.is_available(),
    )

    all_embs = []
    all_labels = []
    all_paths = []

    for images, labels, paths in tqdm(loader, desc=f"Coarse classifier embedding extraction: {split_name}"):
        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in inputs.items()}

        outputs = model.get_image_features(**inputs)
        emb = outputs.pooler_output

        if emb is None:
            raise RuntimeError("SigLIP2 did not return pooler_output")

        emb = F.normalize(emb.float(), dim=-1)

        all_embs.append(emb.cpu())
        all_labels.extend(labels)
        all_paths.extend(paths)

    X = torch.cat(all_embs, dim=0).numpy().astype(np.float32)

    return X, np.array(all_labels).astype(str), np.array(all_paths).astype(str)


# ============================================================
# 4. Generic Model B MLP trainer
# ============================================================

def train_coarse_classifier(X_train_s, y_train, X_val_s, y_val):
    label_encoder = LabelEncoder()
    label_encoder.fit(y_train)

    y_train_enc = label_encoder.transform(y_train)

    train_ds = TensorDataset(
        torch.tensor(X_train_s, dtype=torch.float32),
        torch.tensor(y_train_enc, dtype=torch.long),
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=MLP_BATCH_SIZE,
        shuffle=True,
        drop_last=False,
    )

    model = MLPHead(
        input_dim=X_train_s.shape[1],
        hidden_dim=MLP_HIDDEN_DIM,
        num_classes=len(label_encoder.classes_),
        dropout=MLP_DROPOUT,
    ).to(DEVICE)

    if USE_CLASS_WEIGHT:
        class_weights = make_class_weights(y_train_enc, len(label_encoder.classes_)).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)

    best_state = None
    best_epoch = -1
    best_macro_f1 = -1.0
    patience = 0
    history = []

    for epoch in range(1, MLP_EPOCHS + 1):
        model.train()
        losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

        pred, conf, probs = predict_mlp(model, X_val_s, label_encoder)
        metrics = compute_metrics(y_val, pred)

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            **metrics,
        }

        history.append(row)

        print(
            f"Coarse classifier epoch {epoch:03d} | "
            f"loss={row['train_loss']:.4f} | "
            f"acc={row['accuracy']:.4f} | "
            f"bal_acc={row['balanced_accuracy']:.4f} | "
            f"macro_f1={row['f1_macro']:.4f}"
        )

        if metrics["f1_macro"] > best_macro_f1:
            best_macro_f1 = float(metrics["f1_macro"])
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        if patience >= MLP_PATIENCE:
            print(f"Coarse classifier early stopping | best_epoch={best_epoch} | best_macro_f1={best_macro_f1:.4f}")
            break

    model.load_state_dict(best_state)
    model.eval()

    pred, conf, probs = predict_mlp(model, X_val_s, label_encoder)

    pd.DataFrame(history).to_csv(COARSE_DIR / "training_history.csv", index=False)

    torch.save({
        "model_state_dict": model.state_dict(),
        "label_classes": label_encoder.classes_.tolist(),
        "input_dim": int(X_train_s.shape[1]),
        "hidden_dim": int(MLP_HIDDEN_DIM),
        "dropout": float(MLP_DROPOUT),
        "best_epoch": int(best_epoch),
        "best_macro_f1": float(best_macro_f1),
    }, COARSE_DIR / "coarse_classifier.pt")

    return {
        "model": model,
        "label_encoder": label_encoder,
        "pred": pred,
        "conf": conf,
        "probs": probs,
        "best_epoch": best_epoch,
        "best_macro_f1": best_macro_f1,
    }



# ============================================================
# 5. Specialist dataset
# ============================================================

class SpecialistImageDataset(Dataset):
    def __init__(self, paths, labels, image_processor, label_encoder, transform=None):
        self.paths = [str(p) for p in paths]
        self.labels = [str(x) for x in labels]
        self.image_processor = image_processor
        self.label_encoder = label_encoder
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        enc = self.image_processor(images=image, return_tensors="pt")
        pixel_values = enc["pixel_values"].squeeze(0)
        y = self.label_encoder.transform([self.labels[idx]])[0]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(y, dtype=torch.long),
            "path": self.paths[idx],
        }


# ============================================================
# 6. Specialist inference
# ============================================================

@torch.no_grad()
def predict_specialist_paths(model, image_processor, label_encoder, paths, batch_size=SPECIALIST_BATCH_SIZE):
    model.eval()
    all_probs = []

    for start in range(0, len(paths), batch_size):
        end = min(start + batch_size, len(paths))
        batch_paths = paths[start:end]

        images = [Image.open(str(p)).convert("RGB") for p in batch_paths]
        inputs = image_processor(images=images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(DEVICE)

        outputs = model(pixel_values=pixel_values)
        probs = F.softmax(outputs.logits, dim=1).cpu().numpy()
        all_probs.append(probs)

    probs = np.concatenate(all_probs, axis=0)
    pred_idx = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    pred = label_encoder.inverse_transform(pred_idx)

    return pred, conf, probs


# ============================================================
# 7. Train one specialist
# ============================================================

def train_specialist(group_id, merge_label, group_classes, train_df, val_df, backbone_tag, hf_model_name):
    group_dir = SPECIALIST_DIR / f"group_{group_id:02d}" / backbone_tag
    group_dir.mkdir(parents=True, exist_ok=True)

    specialist_train_df = train_df[train_df["cls_label"].isin(group_classes)].copy().reset_index(drop=True)
    specialist_val_df = val_df[val_df["cls_label"].isin(group_classes)].copy().reset_index(drop=True)

    if len(specialist_train_df) == 0 or len(specialist_val_df) == 0:
        raise ValueError(f"Empty specialist train/val split for group {group_id}")

    y_train = specialist_train_df["cls_label"].astype(str).values
    y_val = specialist_val_df["cls_label"].astype(str).values
    train_paths = specialist_train_df["image_path"].astype(str).values
    val_paths = specialist_val_df["image_path"].astype(str).values

    label_encoder = LabelEncoder()
    label_encoder.fit(y_train)

    num_labels = len(label_encoder.classes_)
    id2label = {i: str(c) for i, c in enumerate(label_encoder.classes_)}
    label2id = {str(c): i for i, c in id2label.items()}

    image_processor = AutoImageProcessor.from_pretrained(hf_model_name)

    model = AutoModelForImageClassification.from_pretrained(
        hf_model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)

    train_ds = SpecialistImageDataset(
        paths=train_paths,
        labels=y_train,
        image_processor=image_processor,
        label_encoder=label_encoder,
        transform=specialist_train_transform if USE_SPECIALIST_AUGMENTATION else None,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=SPECIALIST_BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    y_train_enc = label_encoder.transform(y_train)

    if USE_CLASS_WEIGHT:
        class_weights = make_class_weights(y_train_enc, num_labels).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=SPECIALIST_LR, weight_decay=SPECIALIST_WEIGHT_DECAY)

    best_state = None
    best_epoch = -1
    best_macro_f1 = -1.0
    patience = 0
    history = []

    print("\n" + "=" * 100)
    print("Training specialist:", merge_label)
    print("Classes:", group_classes)
    print("Backbone:", hf_model_name)
    print("Train:", len(specialist_train_df), "| Val:", len(specialist_val_df))
    print("=" * 100)

    for epoch in range(1, SPECIALIST_EPOCHS + 1):
        model.train()
        losses = []

        for batch in tqdm(train_loader, desc=f"Specialist group {group_id} epoch {epoch}", leave=False):
            pixel_values = batch["pixel_values"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(pixel_values=pixel_values).logits
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

        pred, conf, probs = predict_specialist_paths(model, image_processor, label_encoder, val_paths)
        metrics = compute_metrics(y_val, pred)

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            **metrics,
        })

        print(
            f"Specialist group {group_id} epoch {epoch:03d} | "
            f"loss={np.mean(losses):.4f} | "
            f"acc={metrics['accuracy']:.4f} | "
            f"macro_f1={metrics['f1_macro']:.4f}"
        )

        if metrics["f1_macro"] > best_macro_f1:
            best_macro_f1 = float(metrics["f1_macro"])
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        if patience >= SPECIALIST_PATIENCE:
            print(f"Specialist early stopping | best_epoch={best_epoch} | best_macro_f1={best_macro_f1:.4f}")
            break

    model.load_state_dict(best_state)
    model.eval()

    pred, conf, probs = predict_specialist_paths(model, image_processor, label_encoder, val_paths)

    specialist_eval = save_eval_outputs(
        out_dir=group_dir,
        y_true=y_val,
        y_pred=pred,
        y_conf=conf,
        image_paths=val_paths,
        labels_for_cm=label_encoder.classes_.tolist(),
        prefix="specialist_",
    )

    checkpoint_path = group_dir / "specialist.pt"

    torch.save({
        "group_id": int(group_id),
        "merge_label": merge_label,
        "group_classes": list(group_classes),
        "backbone_tag": backbone_tag,
        "hf_model_name": hf_model_name,
        "model_state_dict": model.state_dict(),
        "label_classes": label_encoder.classes_.tolist(),
        "best_epoch": int(best_epoch),
        "best_macro_f1": float(best_macro_f1),
    }, checkpoint_path)

    pd.DataFrame(history).to_csv(group_dir / "training_history.csv", index=False)

    summary = {
        "group_id": group_id,
        "merge_label": merge_label,
        "group_classes": json.dumps(group_classes),
        "backbone_tag": backbone_tag,
        "hf_model_name": hf_model_name,
        "train_n": len(specialist_train_df),
        "val_n": len(specialist_val_df),
        "best_epoch": best_epoch,
        "val_accuracy": specialist_eval["metrics"]["accuracy"],
        "val_balanced_accuracy": specialist_eval["metrics"]["balanced_accuracy"],
        "val_f1_macro": specialist_eval["metrics"]["f1_macro"],
        "checkpoint_path": str(checkpoint_path),
    }

    model.cpu()
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "summary": summary,
        "checkpoint_path": checkpoint_path,
    }



# ============================================================
# 8. Load specialist for final routing
# ============================================================

def load_specialist(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    label_classes = ckpt["label_classes"]
    num_labels = len(label_classes)

    id2label = {i: str(c) for i, c in enumerate(label_classes)}
    label2id = {str(c): i for i, c in id2label.items()}

    image_processor = AutoImageProcessor.from_pretrained(ckpt["hf_model_name"])

    model = AutoModelForImageClassification.from_pretrained(
        ckpt["hf_model_name"],
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    label_encoder = LabelEncoder()
    label_encoder.fit(label_classes)

    return model, image_processor, label_encoder, ckpt


# ============================================================
# 9. Run Model B + specialists
# ============================================================

def run_hierarchical_pipeline():
    print("\n" + "=" * 100)
    print("HIERARCHICAL CLASSIFIER TRAINING")
    print("=" * 100)
    print("Selected encoder:", best_dino_ckpt_path)
    print("Merge groups:", merge_groups)

    set_seed(RANDOM_STATE)

    # --------------------------------------------------------
    # Fresh encoder load and fresh embedding extraction
    # --------------------------------------------------------

    siglip_processor, siglip_model, selected_dino_info = load_adapted_siglip_encoder(best_dino_ckpt_path)

    X_train, _, _ = extract_coarse_embeddings(
        train_df,
        siglip_processor,
        siglip_model,
        "train",
    )

    X_val, _, val_paths = extract_coarse_embeddings(
        val_df,
        siglip_processor,
        siglip_model,
        "val",
    )

    # --------------------------------------------------------
    # Fit a fresh scaler for the coarse classifier
    # --------------------------------------------------------

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_val_s = scaler.transform(X_val).astype(np.float32)

    siglip_model.cpu()
    del siglip_model
    del siglip_processor
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Build coarse labels from the fixed merge mapping
    # --------------------------------------------------------

    train_coarse_df = add_coarse_labels(train_df, leaf_to_coarse)
    val_coarse_df = add_coarse_labels(val_df, leaf_to_coarse)

    train_coarse_df.to_csv(COARSE_DIR / "train_df_with_coarse_label.csv", index=False)
    val_coarse_df.to_csv(COARSE_DIR / "val_df_with_coarse_label.csv", index=False)

    y_train_coarse = train_coarse_df["coarse_label"].astype(str).values
    y_val_coarse = val_coarse_df["coarse_label"].astype(str).values
    y_val_leaf = val_df["cls_label"].astype(str).values

    # --------------------------------------------------------
    # Train coarse classifier
    # --------------------------------------------------------

    coarse_model = train_coarse_classifier(
        X_train_s=X_train_s,
        y_train=y_train_coarse,
        X_val_s=X_val_s,
        y_val=y_val_coarse,
    )

    coarse_classes = coarse_model["label_encoder"].classes_.tolist()

    coarse_eval = save_eval_outputs(
        out_dir=COARSE_DIR,
        y_true=y_val_coarse,
        y_pred=coarse_model["pred"],
        y_conf=coarse_model["conf"],
        image_paths=val_paths,
        labels_for_cm=coarse_classes,
        prefix="coarse_",
    )

    print("\n========== Coarse classifier metrics ==========")
    display(coarse_eval["metrics_df"].T)

    # --------------------------------------------------------
    # Train specialists
    # --------------------------------------------------------

    specialist_infos = {}
    specialist_summaries = []

    for group_id, group_classes in enumerate(merge_groups, start=1):
        merge_label = make_merge_label(group_classes)

        candidate_infos = []

        for backbone_i, backbone_cfg in enumerate(SPECIALIST_BACKBONES):
            set_seed(RANDOM_STATE + group_id * 100 + backbone_i)

            info = train_specialist(
                group_id=group_id,
                merge_label=merge_label,
                group_classes=group_classes,
                train_df=train_df,
                val_df=val_df,
                backbone_tag=backbone_cfg["tag"],
                hf_model_name=backbone_cfg["hf_model_name"],
            )

            candidate_infos.append(info)

        best_info = sorted(
            candidate_infos,
            key=lambda x: (
                x["summary"]["val_f1_macro"],
                x["summary"]["val_balanced_accuracy"],
                x["summary"]["val_accuracy"],
            ),
            reverse=True,
        )[0]

        specialist_infos[merge_label] = best_info
        specialist_summaries.append(best_info["summary"])

    specialist_summary_df = pd.DataFrame(specialist_summaries)
    specialist_summary_df.to_csv(SPECIALIST_DIR / "selected_specialists.csv", index=False)

    print("\n========== Selected specialists ==========")
    display(specialist_summary_df)

    # --------------------------------------------------------
    # Hierarchical routing
    # --------------------------------------------------------

    coarse_pred = np.array(coarse_model["pred"]).astype(str)
    coarse_conf = np.array(coarse_model["conf"]).astype(float)

    hierarchical_pred = coarse_pred.copy()
    hierarchical_conf = coarse_conf.copy()

    hierarchical_route = np.array(["direct_leaf"] * len(val_df), dtype=object)
    hierarchical_specialist_conf = np.full(len(val_df), np.nan, dtype=float)
    hierarchical_specialist_group = np.array([""] * len(val_df), dtype=object)
    hierarchical_specialist_backbone = np.array([""] * len(val_df), dtype=object)

    for merge_label, info in specialist_infos.items():
        routed_idx = np.where(coarse_pred == merge_label)[0]

        print("\nRouting:", merge_label, "| N =", len(routed_idx))

        if len(routed_idx) == 0:
            continue

        routed_paths = val_paths[routed_idx]

        specialist_model, image_processor, label_encoder, ckpt = load_specialist(info["checkpoint_path"])

        specialist_pred, specialist_conf, specialist_probs = predict_specialist_paths(
            model=specialist_model,
            image_processor=image_processor,
            label_encoder=label_encoder,
            paths=routed_paths,
        )

        hierarchical_pred[routed_idx] = specialist_pred
        hierarchical_conf[routed_idx] = coarse_conf[routed_idx] * specialist_conf
        hierarchical_specialist_conf[routed_idx] = specialist_conf
        hierarchical_specialist_group[routed_idx] = merge_label
        hierarchical_specialist_backbone[routed_idx] = ckpt["backbone_tag"]
        hierarchical_route[routed_idx] = "specialist"

        specialist_model.cpu()
        del specialist_model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Hierarchical leaf-level evaluation
    # --------------------------------------------------------

    hierarchical_eval = save_eval_outputs(
        out_dir=HIERARCHICAL_DIR,
        y_true=y_val_leaf,
        y_pred=hierarchical_pred,
        y_conf=hierarchical_conf,
        image_paths=val_paths,
        labels_for_cm=leaf_classes,
        prefix="hierarchical_",
    )

    hierarchical_pred_df = pd.DataFrame({
        "image_path": val_paths,
        "true_leaf": y_val_leaf,
        "true_coarse": y_val_coarse,
        "coarse_pred": coarse_pred,
        "coarse_confidence": coarse_conf,
        "route": hierarchical_route,
        "specialist_group": hierarchical_specialist_group,
        "specialist_backbone": hierarchical_specialist_backbone,
        "specialist_confidence": hierarchical_specialist_conf,
        "hierarchical_pred": hierarchical_pred,
        "hierarchical_confidence": hierarchical_conf,
        "hierarchical_correct": y_val_leaf == hierarchical_pred,
    })

    hierarchical_pred_df.to_csv(HIERARCHICAL_DIR / "hierarchical_predictions.csv", index=False)

    print("\n========== Hierarchical leaf-level metrics ==========")
    display(hierarchical_eval["metrics_df"].T)

    # --------------------------------------------------------
    # Export classifier.pt
    # --------------------------------------------------------

    specialist_heads = {}

    for merge_label, info in specialist_infos.items():
        specialist_ckpt = torch.load(info["checkpoint_path"], map_location="cpu", weights_only=False)

        specialist_heads[merge_label] = {
            "backbone_tag": specialist_ckpt["backbone_tag"],
            "hf_model_name": specialist_ckpt["hf_model_name"],
            "group_classes": list(specialist_ckpt["group_classes"]),
            "label_classes": list(specialist_ckpt["label_classes"]),
            "model_state_dict": {k: v.detach().cpu() for k, v in specialist_ckpt["model_state_dict"].items()},
        }

    classifier_ckpt = {
        "encoder": {
            "model_name": SIGLIP_MODEL_NAME,
            "checkpoint_path": str(best_dino_ckpt_path),
            "checkpoint_state_key": "student_state_dict",
            "dino_epoch": selected_dino_info["epoch"],
            "ssl_train_loss": selected_dino_info["ssl_train_loss"],
        },

        "scaler": {
            "mean": scaler.mean_.astype(np.float32),
            "scale": scaler.scale_.astype(np.float32),
        },

        "coarse_head": {
            "input_dim": int(X_train_s.shape[1]),
            "hidden_dim": int(MLP_HIDDEN_DIM),
            "dropout": float(MLP_DROPOUT),
            "classes": coarse_model["label_encoder"].classes_.tolist(),
            "state_dict": {k: v.detach().cpu() for k, v in coarse_model["model"].state_dict().items()},
        },

        "hierarchy": {
            "leaf_classes": list(leaf_classes),
            "leaf_to_coarse": dict(leaf_to_coarse),
            "merge_groups": [list(group) for group in merge_groups],
        },

        "specialist_heads": specialist_heads,
    }

    classifier_path = OUTPUT_DIR / "hierarchical_classifier.pt"
    torch.save(classifier_ckpt, classifier_path)

    print("\nSaved hierarchical classifier.pt:", classifier_path)

    return {
        "coarse_model": coarse_model,
        "coarse_eval": coarse_eval,
        "specialist_infos": specialist_infos,
        "specialist_summary_df": specialist_summary_df,
        "hierarchical_eval": hierarchical_eval,
        "hierarchical_pred_df": hierarchical_pred_df,
        "classifier_path": classifier_path,
    }



hierarchical_results = run_hierarchical_pipeline()